In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
train_path = "../data/split/train/train.csv"
test_path = "../data/split/test/test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

#drop address
df_train = df_train.drop(columns=['address'])
df_test = df_test.drop(columns=['address'])

# One-hot encode 'region'
df_train = pd.get_dummies(df_train, columns=['region'], prefix='region', drop_first=False)
df_test = pd.get_dummies(df_test, columns=['region'], prefix='region', drop_first=False)

train_dummy_cols = [col for col in df_train.columns if col.startswith("region_")]
for col in train_dummy_cols:
    if col not in df_test.columns:
        df_test[col] = 0  

extra_cols = [col for col in df_test.columns if col.startswith("region_") and col not in train_dummy_cols]
df_test = df_test.drop(columns=extra_cols)

feature_cols = ['area', 'bedrooms', 'bathrooms'] + [col for col in df_train.columns if col.startswith('region_')]
target_col = 'price'

scaler_X = StandardScaler()
df_train[feature_cols] = scaler_X.fit_transform(df_train[feature_cols])
df_test[feature_cols] = scaler_X.transform(df_test[feature_cols])



In [ ]:
X_train = torch.tensor(df_train[feature_cols].to_numpy(dtype=float), dtype=torch.float32)
X_test  = torch.tensor(df_test[feature_cols].to_numpy(dtype=float), dtype=torch.float32)

y_train = torch.tensor(df_train[[target_col]].to_numpy(dtype=float), dtype=torch.float32).reshape(-1, 1) 
y_test  = torch.tensor(df_test[[target_col]].to_numpy(dtype=float), dtype=torch.float32).reshape(-1, 1) 

In [ ]:
model = nn.Linear(in_features=X_train.shape[1], out_features=1)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 10000

for epoch in range(epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

y_pred = model(X_test).detach().numpy()

In [ ]:
print("\n=== EVALUATION ===")
print("R²:", r2_score(y_test.numpy(), y_pred))
print("RMSE:", mean_squared_error(y_test.numpy(), y_pred) ** 0.5)

In [ ]:
print("\n=== MODEL PARAMETERS ===")
print("Weights:", model.weight.detach().numpy())
print("Bias:", model.bias.detach().numpy())